# Desert Farm — Interactive Stommel Diagram

Every process that matters to a desert algae farm, plotted on a
[Stommel diagram](https://en.wikipedia.org/wiki/Stommel_diagram).
Color indicates the dominant energy type.

**Add your own processes** in the "Custom Processes" section below,
then re-run the plot cell to see them on the diagram.

Built with [timeSpace](https://github.com/MDunitz/timeSpace).

In [ ]:
# Install dependencies and clone the package
!pip install -q astropy bokeh colorcet pandas
!git clone -q https://github.com/MDunitz/timeSpace.git
import sys; sys.path.insert(0, 'timeSpace')

In [ ]:
import pandas as pd
import numpy as np
from bokeh.plotting import figure, output_notebook, show, output_file, save
from bokeh.models import (
    ColumnDataSource, Span, Label, HoverTool, Legend, LegendItem
)
from bokeh.resources import CDN

from timeSpace.constants import base_time, base_space, TIME_MARKERS, SPACE_MARKERS
from timeSpace.calculations import create_ellipse_data, classify_process_geometry
from timeSpace.etl import process_magnitude_column
from timeSpace.plotting_helpers import set_fill_alpha

output_notebook()

In [ ]:
# ── Configuration ──────────────────────────────────────────
# Change COLOR_BY to switch between energy type and scale layer
COLOR_BY = "Energy_type"  # or "Scale"

ENERGY_COLORS = {
    "Chemical":   "#0F793D",
    "Radiative":  "#FFCC33",
    "Thermal":    "#CC3333",
    "Mechanical": "#336699",
}

SCALE_COLORS = {
    "Molecular": "#33CCCC",
    "Cellular":  "#009999",
    "Pond":      "#0F793D",
    "Facility":  "#FF9900",
    "Regional":  "#CC9933",
    "Global":    "#996600",
}

X_RANGE = (1e-3, 1e13)
Y_RANGE = (1e-28, 1e22)
N_POINTS = 100

## Load processes

The desert farm dataset ships with the repo. To add your own processes, edit the cell in the next section.

In [ ]:
df = pd.read_csv("timeSpace/docs/desert_farm_processes.csv")

# Apply units (same as the package ETL pipeline)
for col in ["Time_min", "Time_max", "Space_min", "Space_max"]:
    df[col] = df.apply(process_magnitude_column, column=col, axis=1)

print(f"Loaded {len(df)} processes across {df.Scale.nunique()} scales")
df[["Name", "Scale", "Energy_type"]].to_string(index=False)

## Add your own processes

Edit the DataFrame below to add rows. Each row needs:
- **Name**: full name (shown on hover)
- **ShortName**: abbreviated label (shown on diagram)
- **Time_min / Time_max**: seconds (use scientific notation, e.g. `3.2e7`)
- **Space_min / Space_max**: cubic meters
- **Scale**: Molecular, Cellular, Pond, Facility, Regional, or Global
- **Energy_type**: Chemical, Radiative, Thermal, or Mechanical

Then re-run the plot cell below.

In [ ]:
# ── Add your processes here ──────────────────────────────
custom_processes = pd.DataFrame([
    # Example: uncomment and edit these, or add your own rows
    # {"Name": "My process",     "ShortName": "My proc",
    #  "Time_min": 1e2,          "Time_max": 1e5,
    #  "Space_min": 1e-3,        "Space_max": 1e3,
    #  "Scale": "Pond",          "Energy_type": "Chemical"},
])

if len(custom_processes) > 0:
    # Apply units to custom rows
    for col in ["Time_min", "Time_max", "Space_min", "Space_max"]:
        custom_processes[col] = custom_processes.apply(
            process_magnitude_column, column=col, axis=1
        )
    df = pd.concat([df, custom_processes], ignore_index=True)
    print(f"Added {len(custom_processes)} custom processes ({len(df)} total)")
else:
    print("No custom processes added — edit the cell above to add your own")

In [ ]:
# ── Classify geometry and prepare ellipses ─────────────────
df["geometry"] = df.apply(classify_process_geometry, axis=1)

# Only generate ellipse coords for processes with range on both axes
ellipse_mask = df["geometry"] == "ellipse"
if ellipse_mask.any():
    df.loc[ellipse_mask, ["x_coords", "y_coords"]] = (
        df.loc[ellipse_mask, ["Time_min", "Time_max", "Space_min", "Space_max"]]
        .apply(create_ellipse_data, axis=1, result_type="expand")
        .rename(columns={0: "x_coords", 1: "y_coords"})
    )

# Colors and alpha from the package
color_map = ENERGY_COLORS if COLOR_BY == "Energy_type" else SCALE_COLORS
df["color"] = df[COLOR_BY].map(color_map)
df["fill_alpha"] = df.apply(set_fill_alpha, axis=1)
df["label_x"] = np.sqrt(df.Time_min.apply(lambda q: q.value) * df.Time_max.apply(lambda q: q.value))
df["label_y"] = np.sqrt(df.Space_min.apply(lambda q: q.value) * df.Space_max.apply(lambda q: q.value))

print(f"Ready to plot {len(df)} processes ({ellipse_mask.sum()} ellipses, "
      f"{(~ellipse_mask).sum()} degenerate), colored by {COLOR_BY}")

## Stommel Diagram

Click legend entries to toggle energy types on/off. Hover over ellipses for details.

In [ ]:
# ── Build figure ─────────────────────────────────────────
p = figure(
    width=850, height=600,
    x_axis_type="log", y_axis_type="log",
    x_axis_label="Time (s)", y_axis_label="Space (m³)",
    x_range=X_RANGE, y_range=Y_RANGE,
    title="Desert Farm — Processes Across Scale",
    toolbar_location="above",
    tools="pan,wheel_zoom,box_zoom,reset",
)
p.title.text_font_size = "14pt"
p.background_fill_color = "#fafafa"

# Reference grid
for t, txt in TIME_MARKERS.items():
    if X_RANGE[0] <= t <= X_RANGE[1]:
        p.add_layout(Span(location=t, dimension="height",
                          line_color="#ccc", line_dash="dashed", line_width=1))
        p.add_layout(Label(x=t, y=Y_RANGE[1], text=txt,
                           text_font_size="9pt", text_color="#aaa",
                           text_align="center", text_baseline="top"))
for s, txt in SPACE_MARKERS.items():
    if Y_RANGE[0] <= s <= Y_RANGE[1]:
        p.add_layout(Span(location=s, dimension="width",
                          line_color="#ddd", line_dash="dashed", line_width=1))
        p.add_layout(Label(y=s, x=X_RANGE[0]*1.5, text=txt,
                           text_font_size="9pt", text_color="#aaa",
                           text_align="left"))

# Plot by color group
legend_items = []
for group_name in sorted(color_map.keys()):
    gdf = df[df[COLOR_BY] == group_name]
    if gdf.empty:
        continue
    color = color_map[group_name]
    src = ColumnDataSource(data=dict(
        xs=[row.x_coords.tolist() for _, row in gdf.iterrows()],
        ys=[row.y_coords.tolist() for _, row in gdf.iterrows()],
        alpha=gdf.fill_alpha.tolist(),
        name=gdf.Name.tolist(),
        short_name=gdf.ShortName.tolist(),
        scale=gdf.Scale.tolist(),
        energy_type=gdf.Energy_type.tolist(),
        time_min=[r.Time_min.value for _, r in gdf.iterrows()],
        time_max=[r.Time_max.value for _, r in gdf.iterrows()],
        space_min=[r.Space_min.value for _, r in gdf.iterrows()],
        space_max=[r.Space_max.value for _, r in gdf.iterrows()],
        label_x=[r.Time_max.value for _, r in gdf.iterrows()],
        label_y=gdf.label_y.tolist(),
    ))
    patches = p.patches("xs", "ys", source=src,
        fill_color=color, fill_alpha="alpha",
        line_color=color, line_alpha=0.8, line_width=1.5)
    labels = p.text("label_x", "label_y", source=src, text="short_name",
        text_font_size="7pt", text_color=color, text_alpha=0.9,
        text_align="left", text_baseline="middle", x_offset=5)
    legend_items.append(LegendItem(label=group_name, renderers=[patches, labels]))

p.add_tools(HoverTool(tooltips=[
    ("Name", "@name"), ("Scale", "@scale"), ("Energy", "@energy_type"),
    ("Time", "@time_min{%0.1e} → @time_max{%0.1e} s"),
    ("Space", "@space_min{%0.1e} → @space_max{%0.1e} m³"),
], formatters={
    "@time_min": "printf", "@time_max": "printf",
    "@space_min": "printf", "@space_max": "printf",
}))

legend = Legend(items=legend_items, location="top_left",
    label_text_font_size="10pt", click_policy="hide",
    title="Dominant Energy" if COLOR_BY == "Energy_type" else "Scale Layer",
    title_text_font_size="11pt", title_text_font_style="bold")
p.add_layout(legend, "right")

show(p)

## Export

Run the cell below to save your diagram as an HTML file you can download.

In [ ]:
# Save to HTML — download from Colab file browser (left sidebar)
output_file("my_stommel.html", title="My Stommel Diagram")
save(p)
print("Saved → my_stommel.html (check the file browser on the left)")